### Want to visualize behavior of a trained network (IPPO), against either a leader or a follower. 
Steps: 
- load params from checkpoint
- run model/env
- track trajectories
- plot

### Load parameters

In [1]:
%load_ext autoreload
%autoreload 2

import jax
import jax.numpy as jnp
import pickle
import imageio
from jaxmarl.viz.toy_coop_jitted_visualizer import render_fn
from jaxmarl.environments.toy_coop.coop_foraging_fixed_other import leader_action, follower_action
from jaxmarl.environments import CoopForagingFixedOther, CoopForaging
from jaxmarl.environments import spaces

import sys
sys.path.append("/src/tasks_and_others") 
from baselines.CEC.ippo_general import ActorCriticRNN, ScannedRNN

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
def gen_heldout(env):
    def gen_held_out_toycoop(runner_state, unused):
        (i,) = runner_state
        key = jax.random.key(i)
        state = env.custom_reset_fn(key, random_reset=True)
        res = (state.agent_pos, state.goal_pos, state.other_goal_pos)
        carry = (i+1,)
        return carry, res
    
    carry, res = jax.lax.scan(gen_held_out_toycoop, (0,), jnp.arange(100), 100)
    ho_agent_pos, ho_goal_pos, ho_other_goal_pos = res
    
    # Set the held-out states in the environment
    env.held_out_agent_pos = ho_agent_pos
    env.held_out_goal_pos = ho_goal_pos
    env.held_out_other_goal_pos = ho_other_goal_pos
    return env

# Load the trained model
def load_model(filepath):
    with open(filepath, "rb") as f:
        checkpoint = pickle.load(f)
    return checkpoint["params"]

# Run the model on the environment
def run_model_on_task(env, model_params, rng, num_steps=100, other_agent_type=0):
    # Initialize the network
    config = {
        "GRU_HIDDEN_DIM": 128,
        "FC_DIM_SIZE": 128,
        "LSTM": True,
        "GRAPH_NET": False,
        "ENV_NAME": "CoopForagingFixedOther",
    }
    network = ActorCriticRNN(env.num_actions, config=config)
    hidden_state = ScannedRNN.initialize_carry(1, config["GRU_HIDDEN_DIM"])

    # debugging code, randomly initialize the model params
    # rng, _rng = jax.random.split(rng)


    # flattened_obs_dim = 1
    # for dim in env.observation_space(env.agents[0]).shape:
    #     flattened_obs_dim *= dim
    # init_x = (
    #     jnp.zeros(
    #         (1, 1, flattened_obs_dim)
    #     ),
    #     jnp.zeros((1, 1)),
    #     jnp.zeros((1, 1, 2, 2)).astype(jnp.int32)
    # )
    # init_hstate = ScannedRNN.initialize_carry(1, config["GRU_HIDDEN_DIM"])
    # model_params = network.init(_rng, init_hstate, init_x)


    # Reset the environment
    rng, reset_rng = jax.random.split(rng)
    obs, state = env.reset_with_other_agent(reset_rng, other_agent_type_idx=other_agent_type)

    # Store frames for visualization
    frames = []
    total_rewards = 0
    for step in range(num_steps):
        # Prepare the input for the network
        obs_batch = jnp.expand_dims(obs[env.agent_id], axis=(0, 1))
        dones = jnp.zeros((1, 1))  # No episode termination during this run
        agent_positions = jnp.expand_dims(state.agent_pos, axis=(0, 1))

        # Run the model
        hidden_state, pi, _ = network.apply(model_params, hidden_state, (obs_batch, dones, agent_positions))
        # print(pi.probs)
        rng, _rng = jax.random.split(rng)
        action = pi.sample(seed=_rng)

        # Step the environment
        rng, step_rng = jax.random.split(rng)
        # print(jnp.squeeze(action[0]))
        obs, state, rewards, _, _ = env.step(step_rng, state, {"agent_0": jnp.squeeze(action[0])})
        total_rewards += rewards["agent_0"]
        # Render the current state
        frame = render_fn(state, size=env.env.height)
        frames.append(frame)
    print(f"Total rewards: {total_rewards}")
    return frames

# Save frames as a GIF
def save_gif(frames, filename):
    frames = [frame.astype("uint8") for frame in frames]
    imageio.mimsave(filename, frames, fps=5)


In [10]:
model_filepath = "/src/tasks_and_others/ckpts/ippo/CoopForagingFixedOther/ikTrue/reset_all/graphFalse/seed0_ckpt0_improved.pkl"  # Replace with the actual path to the model
model_params  = load_model(model_filepath)

SEED = 3
rng = jax.random.PRNGKey(SEED)
# Initialize the environment
env_leader = CoopForagingFixedOther(random_reset=True, size=8)
env_leader = gen_heldout(env_leader)
env_follower = CoopForagingFixedOther(random_reset=True, size=8)
env_follower = gen_heldout(env_follower)


# Set the fixed agent type to leader (0) for the first run

# Run the model against a leader

print("Running model against a leader...")
frames_leader = run_model_on_task(env_leader, model_params, rng, num_steps=100, other_agent_type=0)
save_gif(frames_leader, f"trajectories_seed_{SEED}_leader_goal0.gif")
print("Saved trajectories against a leader as 'trajectories_leader_goal0.gif'.")

print("Running model against a leader...")
frames_leader = run_model_on_task(env_leader, model_params, rng, num_steps=100, other_agent_type=1)
save_gif(frames_leader, f"trajectories_seed_{SEED}_leader_goal1.gif")
print("Saved trajectories against a leader as 'trajectories_leader_goal1.gif'.")

# Run the model against a follower
print("Running model against a follower...")
frames_follower = run_model_on_task(env_follower, model_params, rng, num_steps=100, other_agent_type=2)
save_gif(frames_follower, f"trajectories_seed_{SEED}_follower.gif")
print("Saved trajectories against a follower as 'trajectories_follower.gif'.")

Running model against a leader...
Total rewards: 173.0
Saved trajectories against a leader as 'trajectories_leader_goal0.gif'.
Running model against a leader...
Total rewards: 194.0
Saved trajectories against a leader as 'trajectories_leader_goal1.gif'.
Running model against a follower...
Total rewards: 194.0
Saved trajectories against a follower as 'trajectories_follower.gif'.
